In [1]:

import numpy as np

NON1_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_lfcc_non-speech_1.npy"
NON2_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_lfcc_non-speech_2.npy"
SPEECH_PATH = "/home/quochuy/Downloads/data_train_fpt/train_ail_data/mini_lfcc_speech.npy"

X_non1 = np.load(NON1_PATH, mmap_mode="r")
X_non2 = np.load(NON2_PATH, mmap_mode="r")
X_speech = np.load(SPEECH_PATH, mmap_mode="r")

print("Non1 :", X_non1.shape)
print("Non2 :", X_non2.shape)
print("Speech:", X_speech.shape)


Non1 : (1094415, 60)
Non2 : (168465, 60)
Speech: (1253962, 60)


In [2]:
# %%
X_non = np.concatenate([
    X_non1[:, :60],
    X_non2[:, :60]
], axis=0)

X_speech = X_speech[:, :60]

print( X_non.shape, X_speech.shape)


(1262880, 60) (1253962, 60)


In [3]:
# %%
X = np.vstack([X_non, X_speech]).astype(np.float32)

y = np.hstack([
    np.zeros(len(X_non), dtype=np.int8),
    np.ones(len(X_speech), dtype=np.int8)
])

print("X:", X.shape)
print("y:", np.bincount(y))


X: (2516842, 60)
y: [1262880 1253962]


In [4]:
# %%
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)


In [5]:
# %%
def run_cv(model_fn, X, y, k=5, batch_size=200_000):
    skf = StratifiedKFold(
        n_splits=k,
        shuffle=True,
        random_state=42
    )

    metrics = {"acc": [], "prec": [], "rec": [], "f1": [], "auc": []}

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n===== FOLD {fold}/{k} =====")

        X_train = X[train_idx]
        y_train = y[train_idx]

        X_val = X[val_idx]
        y_val = y[val_idx]

        model = model_fn()
        model.fit(X_train, y_train)

        y_pred_all, y_prob_all = [], []

        for i in range(0, len(X_val), batch_size):
            Xb = X_val[i:i + batch_size]
            y_pred_all.append(model.predict(Xb))
            y_prob_all.append(model.predict_proba(Xb)[:, 1])

        y_pred = np.concatenate(y_pred_all)
        y_prob = np.concatenate(y_prob_all)

        metrics["acc"].append(accuracy_score(y_val, y_pred))
        metrics["prec"].append(precision_score(y_val, y_pred))
        metrics["rec"].append(recall_score(y_val, y_pred))
        metrics["f1"].append(f1_score(y_val, y_pred))
        metrics["auc"].append(roc_auc_score(y_val, y_prob))

        print(
            f"Acc={metrics['acc'][-1]:.4f} | "
            f"Prec={metrics['prec'][-1]:.4f} | "
            f"Rec={metrics['rec'][-1]:.4f} | "
            f"F1={metrics['f1'][-1]:.4f} | "
            f"AUC={metrics['auc'][-1]:.4f}"
        )

    print("\n===== CV MEAN (5-FOLD) =====")
    for k in metrics:
        print(f"{k.upper():5s}: {np.mean(metrics[k]):.4f}")


In [6]:
from sklearn.ensemble import RandomForestClassifier

def rf_model():
    return RandomForestClassifier(
        n_estimators=80,
        max_depth=15,
        min_samples_leaf=10,
        n_jobs=4,
        random_state=42
    )

run_cv(rf_model, X, y, k=5)



===== FOLD 1/5 =====
Acc=0.9441 | Prec=0.9383 | Rec=0.9503 | F1=0.9443 | AUC=0.9866

===== FOLD 2/5 =====
Acc=0.9441 | Prec=0.9377 | Rec=0.9509 | F1=0.9443 | AUC=0.9865

===== FOLD 3/5 =====
Acc=0.9443 | Prec=0.9380 | Rec=0.9510 | F1=0.9445 | AUC=0.9866

===== FOLD 4/5 =====
Acc=0.9437 | Prec=0.9373 | Rec=0.9507 | F1=0.9439 | AUC=0.9865

===== FOLD 5/5 =====
Acc=0.9446 | Prec=0.9382 | Rec=0.9515 | F1=0.9448 | AUC=0.9868

===== CV MEAN (5-FOLD) =====
ACC  : 0.9442
PREC : 0.9379
REC  : 0.9509
F1   : 0.9443
AUC  : 0.9866


In [ ]:
# %%
from xgboost import XGBClassifier
    
def xgb_model():
    return XGBClassifier(
        tree_method="hist",
        device="cuda",
        max_depth=10,
        n_estimators=300,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

run_cv(xgb_model, X, y, k=5)
